# Wan2.2 I2V — ComfyUI on Google Colab
## 模型: Wan2.2-I2V-A14B Q6_K GGUF + Lightx2v 4-steps LoRA

**⚠️ 預備工作**:
1. `Runtime → Restart runtime` (全新環境)
2. `Runtime → Change runtime type → GPU` (T4 或以上)
3. **你需要一個 zrok token** → https://dashboard.zrok.io


In [ ]:
# ============================================================
# A) 系統套件
# ============================================================
!apt-get -y update -qq
!apt-get -y install -qq ffmpeg aria2
print('✅ System packages ready')

In [ ]:
# ============================================================
# B) 安裝 ComfyUI + Python 依賴
# ============================================================
%cd /content
!rm -rf /content/ComfyUI
!git clone https://github.com/comfyanonymous/ComfyUI.git
%cd /content/ComfyUI
!git pull

!pip -q install -r requirements.txt
!pip -q install imageio-ffmpeg ffmpeg-python
print('✅ ComfyUI installed')

In [ ]:
# ============================================================
# C1) 安裝 ComfyUI-GGUF (GGUF Loader — 必裝)
# ============================================================
%cd /content/ComfyUI/custom_nodes
!rm -rf ComfyUI-GGUF
!git clone https://github.com/city96/ComfyUI-GGUF.git

!pip -q install gguf
!pip -q install -r /content/ComfyUI/custom_nodes/ComfyUI-GGUF/requirements.txt || true
print('✅ ComfyUI-GGUF installed')

In [ ]:
# ============================================================
# C2) 安裝 VideoHelperSuite (進度條 + 預覽)
# ============================================================
%cd /content/ComfyUI/custom_nodes
!rm -rf ComfyUI-VideoHelperSuite
!git clone https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git
print('✅ VideoHelperSuite installed')

In [ ]:
# ============================================================
# D) 下載模型 (aria2c 多線程)
# ============================================================
%cd /content/ComfyUI

# D1: Q6 GGUF UNet (High Noise + Low Noise)
!aria2c -x 16 -s 16 -k 1M \
  -d 'models/unet/' \
  -o 'Wan2.2-I2V-A14B-HighNoise-Q6_K.gguf' \
  'https://huggingface.co/QuantStack/Wan2.2-I2V-A14B-GGUF/resolve/main/HighNoise/Wan2.2-I2V-A14B-HighNoise-Q6_K.gguf'
print('✅ HighNoise UNet downloaded')

!aria2c -x 16 -s 16 -k 1M \
  -d 'models/unet/' \
  -o 'Wan2.2-I2V-A14B-LowNoise-Q6_K.gguf' \
  'https://huggingface.co/QuantStack/Wan2.2-I2V-A14B-GGUF/resolve/main/LowNoise/Wan2.2-I2V-A14B-LowNoise-Q6_K.gguf'
print('✅ LowNoise UNet downloaded')

# D2: VAE
!aria2c -x 16 -s 16 -k 1M \
  -d 'models/vae/' \
  -o 'Wan2.1_VAE.safetensors' \
  'https://huggingface.co/QuantStack/Wan2.2-I2V-A14B-GGUF/resolve/main/VAE/Wan2.1_VAE.safetensors?download=true'
print('✅ VAE downloaded')

# D3: UMT5 Text Encoder (Q3)
!aria2c -x 16 -s 16 -k 1M \
  -d 'models/text_encoders/' \
  -o 'umt5-xxl-encoder-Q3_K_M.gguf' \
  'https://huggingface.co/city96/umt5-xxl-encoder-gguf/resolve/main/umt5-xxl-encoder-Q3_K_M.gguf?download=true'
print('✅ UMT5 text encoder downloaded')

# D4: LoRA
!aria2c -x 16 -s 16 -k 1M \
  -d 'models/loras/' \
  -o 'high_noise_model.safetensors' \
  'https://huggingface.co/lightx2v/Wan2.2-Lightning/resolve/main/Wan2.2-I2V-A14B-4steps-lora-rank64-Seko-V1/high_noise_model.safetensors?download=true'

!aria2c -x 16 -s 16 -k 1M \
  -d 'models/loras/' \
  -o 'low_noise_model.safetensors' \
  'https://huggingface.co/lightx2v/Wan2.2-Lightning/resolve/main/Wan2.2-I2V-A14B-4steps-lora-rank64-Seko-V1/low_noise_model.safetensors?download=true'
print('✅ LoRA files downloaded')

In [ ]:
# ============================================================
# 驗證模型檔案
# ============================================================
!ls -lh models/unet/
!ls -lh models/vae/
!ls -lh models/text_encoders/
!ls -lh models/loras/

In [ ]:
# ============================================================
# E) 記憶體優化
# ============================================================
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128,expandable_segments:True'

!nvidia-smi

In [ ]:
# ============================================================
# F) 啟動 ComfyUI + zrok 公開 URL
# ============================================================
# Step 1: 啟動 ComfyUI (背景)
%cd /content/ComfyUI
!nohup python main.py --listen 0.0.0.0 --port 8188 --lowvram > /content/comfy.log 2>&1 &
print('⏳ ComfyUI 啟動中...')
import time
time.sleep(5)

# Step 2: 安裝 zrok
!command -v zrok || curl -sSf https://get.openziti.io/install.bash | bash -s zrok

# Step 3: 設定 PATH
import os
os.environ['PATH'] += ':/usr/local/bin:/usr/bin:/bin'

# Step 4: 重置 zrok
!zrok disable || true
time.sleep(2)

In [ ]:
# ============================================================
# ⚠️  填入你的 zrok token
# 取得: https://dashboard.zrok.io
# ============================================================
from google.colab import output
output.clear()

ZROK_TOKEN = ''  # 👈 在這裡填入你的 zrok token

if not ZROK_TOKEN:
    print('❌ 請先到 https://dashboard.zrok.io 取得 token 並填入上方 ZROK_TOKEN')
else:
    !zrok enable $ZROK_TOKEN
    import time
    time.sleep(3)
    print('🌍 公開 URL 即將出現:')
    !zrok share public http://127.0.0.1:8188

In [ ]:
# ============================================================
# G) Workflow 說明 (在 Colab 裡操作 ComfyUI)
# ============================================================
print('📋 Workflow 節點設定:')
print('   - UnetLoaderGGUF: Wan2.2-I2V-A14B-HighNoise-Q6_K.gguf')
print('   - CLIPLoaderGGUF: umt5-xxl-encoder-Q3_K_M.gguf')
print('   - VAELoader: Wan2.1_VAE.safetensors')
print('   - LoraLoaderModelOnly: high_noise_model.safetensors (strength=1.0)')
print('   - WanImageToVideo: 480x832, 97 frames, 1 seed')
print('   - KSamplerAdvanced: 4 steps, Euler, simple')
print()
print('📎 使用方法:')
print('   1. 開啟 ComfyUI WebUI (zrok URL)')
print('   2. 手動建立 workflow 或 Load workflow JSON')
print('   3. 上傳圖片到 LoadImage 節點')
print('   4. 點 Queue')